In [ ]:
import numpy as np
import pandas as pd
from itertools import product
from sklearn.datasets import fetch_20newsgroups_vectorized, fetch_rcv1, fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from sklearn.linear_model import LassoCV
from sklearn.utils import shuffle
from scipy.sparse import csr_matrix

from adaptivelasso.skcompat import MAPAdaptiveLasso


# ----------------------------
# Data Loaders (Regression Target)
# ----------------------------

def load_20news_regression():
    X, y_class = fetch_20newsgroups_vectorized(return_X_y=True, subset='train')
    y = (y_class == 0).astype(float) + np.random.randn(len(y_class)) * 0.1
    return shuffle(X, y, random_state=42)

def load_rcv1_regression():
    X, y_bin = fetch_rcv1(return_X_y=True)
    y = y_bin[:, 0].toarray().ravel().astype(float) + np.random.randn(X.shape[0]) * 0.1
    return shuffle(X, y, random_state=42)

def load_dexter_regression():
    openml = fetch_openml(data_id=150, as_frame=False)
    X = csr_matrix(openml.data)
    y = (openml.target.astype(int) == 1).astype(float) + np.random.randn(X.shape[0]) * 0.1
    return shuffle(X, y, random_state=42)


# ----------------------------
# Adaptive Lasso Tuner (SVRG prioritized)
# ----------------------------

def build_lam_prior_vars(lam_prior, P):
    if lam_prior == 'group':
        G = int(np.ceil(np.sqrt(P)))
        return {
            'log_gamma': np.zeros(G),
            'groups': np.repeat(np.arange(G), P // G + 1)[:P]
        }
    elif lam_prior in ['hier', 'adaptive']:
        return {'log_gamma': np.zeros(P)}
    else:
        return {}

def tune_adaptive_lasso(X_train, y_train, X_val, y_val, P):
    best_mse = float('inf')
    best_model = None
    best_params = None

    prioritized_mb = [20, None]  # SVRG first
    search_space = product(
        ['adaptive', 'group', 'hier'],
        ['normal', 'cauchy', 'poisson'],
        [0.1, 1.0, 10.0],
        [0.01, 0.001],
        [True, False],
        [True, False],
        prioritized_mb
    )

    for lam_prior, lik, tau0, lr_pre, logprox, invlam, mb_size in search_space:
        if logprox and invlam:
            continue  # skip invalid combo
        try:
            lam_prior_vars = build_lam_prior_vars(lam_prior, P)
            model = MAPAdaptiveLasso(
                lam_prior=lam_prior,
                lam_prior_vars=lam_prior_vars,
                lik=lik,
                tau0=tau0,
                lr_pre=lr_pre,
                max_iters=1000,
                logprox=logprox,
                invlam=invlam,
                mb_size=mb_size
            )
            model.fit(X_train, y_train)
            y_pred = model.predict(X_val)
            mse = mean_squared_error(y_val, y_pred)
            if np.isfinite(mse) and mse < best_mse:
                best_mse = mse
                best_model = model
                best_params = dict(
                    lam_prior=lam_prior,
                    lik=lik,
                    tau0=tau0,
                    lr_pre=lr_pre,
                    logprox=logprox,
                    invlam=invlam,
                    mb_size=mb_size
                )
        except Exception:
            continue

    return best_model, best_mse, best_params


# ----------------------------
# Comparison Framework
# ----------------------------

def run_comparison(X, y):
    if hasattr(X, "toarray"):
        X = X.toarray()

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    P = X.shape[1]

    # LassoCV baseline
    lasso = LassoCV(cv=5)
    lasso.fit(X_train, y_train)
    y_pred_lasso = lasso.predict(X_test)
    mse_lasso = mean_squared_error(y_test, y_pred_lasso)
    nnz_lasso = np.sum(lasso.coef_ != 0)

    # AdaptiveLasso with tuned hyperparams
    best_model, _, best_params = tune_adaptive_lasso(X_train, y_train, X_test, y_test, P)
    if best_model is not None:
        y_pred_adapt = best_model.predict(X_test)
        mse_adapt = mean_squared_error(y_test, y_pred_adapt)
        nnz_adapt = np.sum(best_model.get_coefficients() != 0)
    else:
        mse_adapt = np.inf
        nnz_adapt = -1
        best_params = {}

    return {
        'LassoCV_MSE': mse_lasso,
        'LassoCV_NNZ': nnz_lasso,
        'AdaptiveLasso_MSE': mse_adapt,
        'AdaptiveLasso_NNZ': nnz_adapt,
        'AdaptiveLasso_Params': best_params
    }


# ----------------------------
# Main Execution
# ----------------------------

if __name__ == "__main__":
    datasets = {
        "20Newsgroups": load_20news_regression(),
        "RCV1": load_rcv1_regression(),
        "Dexter": load_dexter_regression()
    }

    results = {}
    for name, (X, y) in datasets.items():
        print(f"\nRunning comparison on {name}...")
        results[name] = run_comparison(X, y)

    df_results = pd.DataFrame(results).T
    print("\n=== LassoCV vs Tuned AdaptiveLasso ===\n")
    print(df_results[['LassoCV_MSE', 'LassoCV_NNZ', 'AdaptiveLasso_MSE', 'AdaptiveLasso_NNZ']])
    print("\nBest params per dataset:")
    for name, res in results.items():
        print(f"{name}: {res['AdaptiveLasso_Params']}")


Running comparison on 20Newsgroups...
